# Подсказки к Задаче 2

Здесь собраны **фрагменты кода и пояснения** для дополнительных пунктов на 10 баллов.

Каждая секция — отдельный бонусный пункт. Выберите **любые 3** из 5.

---

## Бонус 1: Dropout + Early Stopping (+1 балл)

### Что это

- **Dropout** — случайно «выключает» часть нейронов во время обучения, чтобы сеть не запоминала данные наизусть (переобучение)
- **Early Stopping** — останавливает обучение, когда ошибка на валидации перестаёт падать

### Как добавить в модель

```python
from tensorflow.keras.callbacks import EarlyStopping

model = keras.Sequential([
    layers.Dense(32, activation="relu", input_shape=(n_features,)),
    layers.Dropout(0.3),          # <-- выключает 30% нейронов случайно
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),          # <-- ещё один dropout
    layers.Dense(1, activation="sigmoid"),
])

early_stop = EarlyStopping(
    monitor="val_loss",   # следим за ошибкой на валидации
    patience=10,          # ждём 10 эпох без улучшения
    restore_best_weights=True  # возвращаем лучшие веса
)

history = model.fit(
    X_train, y_train,
    epochs=200,  # ставим больше — early stopping остановит раньше
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Обучение остановлено на эпохе {len(history.history['loss'])}")
```

**Что написать**: объясните, зачем Dropout и Early Stopping нужны, и покажите, как изменились графики.

---

## Бонус 2: Мультиклассовая классификация (+1 балл)

### Что меняется

Вместо 2 классов (позитив / негатив) — 3 класса (позитив / негатив / нейтрал).

| Что | Бинарная | Мультиклассовая |
|-----|----------|-----------------|
| Выходной слой | `Dense(1, sigmoid)` | `Dense(3, softmax)` |
| Loss | `binary_crossentropy` | `sparse_categorical_crossentropy` |
| Метки | 0 или 1 | 0, 1 или 2 |

### Фрагмент кода

```python
# Берём ВСЕ данные (включая нейтральные)
X_all = vectorizer.fit_transform(df["text"]).toarray()
y_all = df["sentiment"].values  # 0, 1, 2

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)

model_multi = keras.Sequential([
    layers.Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(3, activation="softmax"),  # <-- 3 класса!
])

model_multi.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # <-- другой loss!
    metrics=["accuracy"],
)

model_multi.fit(X_train, y_train, epochs=80, batch_size=8,
                validation_split=0.2, verbose=0)
```

**Что написать**: чем отличается softmax от sigmoid, зачем нужен другой loss.

---

## Бонус 3: Сравнение архитектур (+1 балл)

### Идея

Обучите 2–3 модели с разной структурой и сравните результаты.

### Пример: три варианта

```python
configs = {
    "Маленькая (16)":   [16, 1],
    "Средняя (32→16)":  [32, 16, 1],
    "Большая (64→32→16)": [64, 32, 16, 1],
}

results = []

for name, layer_sizes in configs.items():
    m = keras.Sequential()
    m.add(layers.Dense(layer_sizes[0], activation="relu",
                       input_shape=(X_train.shape[1],)))
    for size in layer_sizes[1:-1]:
        m.add(layers.Dense(size, activation="relu"))
    m.add(layers.Dense(1, activation="sigmoid"))

    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(X_train, y_train, epochs=80, batch_size=8, verbose=0)

    y_pred = (m.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    results.append({
        "Архитектура": name,
        "F1": round(f1_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
    })

pd.DataFrame(results)
```

**Что написать**: какая архитектура лучше и почему (больше нейронов ≠ всегда лучше).

---

## Бонус 4: Forward pass на numpy (+1 балл)

### Идея

Достать веса из обученной keras-модели и вручную вычислить предсказание — без `model.predict()`.

Это показывает, что нейросеть — просто математика: умножение матриц + активация.

### Фрагмент кода

```python
# Достаём веса из обученной модели
all_weights = []
for layer in model.layers:
    w = layer.get_weights()
    if w:
        all_weights.append(w)  # [W, b] для каждого Dense-слоя

# Прямой проход вручную
def forward_numpy(X, weights):
    out = X
    for i, (W, b) in enumerate(weights):
        out = out @ W + b
        if i < len(weights) - 1:
            out = np.maximum(0, out)  # ReLU
        else:
            out = 1 / (1 + np.exp(-out))  # Sigmoid
    return out

weights = [(w, b) for w, b in all_weights]
y_numpy = forward_numpy(X_test, weights)

# Сравниваем с keras
y_keras = model.predict(X_test, verbose=0)

print("Максимальная разница:", np.max(np.abs(y_numpy - y_keras)))
# Должно быть очень маленькое число (~1e-7)
```

**Что написать**: объясните, что происходит на каждом шаге forward pass (умножение, bias, активация).

---

## Бонус 5: Embedding вместо TF-IDF (+1 балл)

### Идея

Вместо TF-IDF использовать слой `Embedding`, который сам учится представлять слова как вектора (как в лекции 2, notebook_2).

### Фрагмент кода

```python
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Токенизация: слова → числа
tokenizer = Tokenizer(num_words=500)
tokenizer.fit_on_texts(df_binary["text"])
X_seq = tokenizer.texts_to_sequences(df_binary["text"])
X_pad = pad_sequences(X_seq, maxlen=20, padding="post")

# Делим данные
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_pad, df_binary["sentiment"].values,
    test_size=0.3, random_state=42, stratify=df_binary["sentiment"]
)

# Модель с Embedding
model_emb = keras.Sequential([
    layers.Embedding(input_dim=500, output_dim=16, input_length=20),
    layers.GlobalAveragePooling1D(),  # усредняет эмбеддинги всех слов
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model_emb.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_emb.fit(X_train_e, y_train_e, epochs=80, batch_size=8,
              validation_split=0.2, verbose=0)
```

**Что написать**: чем Embedding отличается от TF-IDF (учится из данных vs фиксированная формула).

---

## Чеклист для сдачи

### Уровень 1 (до 7 баллов)

- [ ] Данные подготовлены (TF-IDF + train/test)
- [ ] Модель на keras построена и обучена
- [ ] Метрики: F1, Precision, Recall
- [ ] Графики loss/accuracy по эпохам
- [ ] Текстовое объяснение (что делает каждый слой, что такое loss, fit, predict)

### Уровень 2 (до 10 баллов)

Всё из уровня 1 **плюс любые 3 из**:

- [ ] Dropout + Early Stopping (+ объяснение)
- [ ] Мультиклассовая классификация (softmax, 3 класса)
- [ ] Сравнение 2–3 архитектур (таблица метрик + вывод)
- [ ] Forward pass на numpy (совпадение с predict)
- [ ] Embedding вместо TF-IDF